In [ ]:
import re

def parse_log_file(log_file_path):
    total_operations = 0
    overshoot_operations = 0
    
    with open(log_file_path, 'R', encoding='Utf 8') as file:
        lines = file.readlines()
    
    i = 0
    while i < len(lines):
        # Test new experiment starts
        if "==== Experiment " in lines[i] and "Start ====" in lines[i]:
            i += 1
            continue
        
        # Find state action reagent pair part
        if "Status Action Reagent pair:" in lines[i]:
            i += 1
            while i < len(lines) and lines[i].strip() and not "Experiment ends" in lines[i]:
                # Analyze every step
                match = re.match(r"  Step \d+: State = \[([^\]]+)\], Action = ([0-9.]+), Reagent = (\w+)", lines[i])
                if match:
                    total_operations += 1
                    state_str, action, reagent = match.groups()
                    state_values = [float(x) for x in state_str.split()]
                    current_ph = state_values[0]
                    target_ph = state_values[1]
                    
                    # Get the pH of the previous state (if present)
                    if i > 1:
                        prev_line = lines[i - 1]
                        prev_match = re.match(r"  Step \d+: State = \[([^\]]+)\]", prev_line)
                        if prev_match:
                            prev_state_values = [float(x) for x in prev_match.group(1).split()]
                            prev_ph = prev_state_values[0]
                            # Check for overshoot: the product of the previous state and the target has the opposite sign of the product of the current state and the target, and has not reached near the target p h (plus or minus 0.1)
                            if (prev_ph - target_ph) * (current_ph - target_ph) < 0 and abs(current_ph - target_ph) > 0.1:
                                overshoot_operations += 1
                
                i += 1
            i += 1  # Skip the "end of experiment" line
        else:
            i += 1
    
    overshoot_ratio = (overshoot_operations / total_operations * 100) if total_operations > 0 else 0
    return total_operations, overshoot_operations, overshoot_ratio

# Usage example
log_file_path = r"C:\users\ZSY\Desktop\code\ph\powerful化前神经网络同样的实验.txt"  # Replace with your log file path
total_ops, overshoot_ops, overshoot_ratio = parse_log_file(log_file_path)

print(f"Total number of experimental operations: {total_ops}")
print(f"The number of experimental operations that caused the overshoot: {overshoot_ops}")
print(f"Overshoot ratio: {overshoot_ratio:.2f}%")

总实验操作数: 38104
引起过冲的实验操作数: 10835
过冲比例: 28.44%
